# Ch6 固定 P4 四色垃圾桶自动分拣

在 Jetson 的 JupyterLab 中运行本 notebook，本机浏览器只作操作界面。先预览/演练，真机运行前核对 `bin_map.json`。运行定义单元不会移动机械臂。完整说明见 README.md。


In [ ]:
from pathlib import Path
import sys
BASE = Path.cwd()
if not (BASE / 'week6_color_sorting.py').is_file():
    raise FileNotFoundError('请在 week6_p4 文件夹中打开 notebook')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
from sorting_core import load_config
from week6_color_sorting import create_panel
config = load_config(BASE / 'bin_map.json')
print('默认演练；物块每轮放在固定 P4；自动识别颜色，无需下拉选择颜色。')
print('桶位 Roll 尚未填入时，可以预览/演练，但不能连接真机。')


## 参数检查

所有角度在配置中为度，发给 SDK 时只转换一次。不要把 week4 的堆叠放置高度当作 P4 抓取高度；不要直接把异常 Roll 填入运动位姿。编辑本新建配置后，停止旧面板并重新运行参数加载单元。


In [ ]:
for color, item in config['bin_map'].items():
    print(color, item['label'], item['pose'], 'raw_roll=', item['raw_roll_deg'])
print('P4:', config['p4'])


## 实时预览和自动分拣

点击启动摄像头，用 x/y/width/height 圈住 P4 并应用 ROI；演练模式下点击开始自动分拣。每轮只放一个物块。分拣后必须在动作结束之后连续看到 P4 空区，才会重新准备；BUSY 遮挡不计为空区。

真机：先完成示教/检查并更新配置，再连接真机、回观察位、重新设置 P4 ROI、开始自动分拣。停止当前动作会取消剩余轨迹并保持位置，真实表现仍需现场检查。


In [ ]:
runtime = create_panel(config)


## 数据记录

每轮识别截图、阶段事件自动保存到 results。现场观察后，在面板填真实颜色、实际投放桶、成功/失败和备注，点击记录本轮实测。程序 done 不自动算实际分拣成功。课程 PDF 建议 8 次中至少 7 次正确；报告模板写至少 6 次，以教师最终要求为准。


In [ ]:
print('本次结果目录：', runtime.output)
print('当前状态：', runtime.gate.state)


## 结束

点击关闭预览，或运行下面单元，等待动作线程制停并关闭串口后再关闭内核。停止后不自动回零或松爪。


In [ ]:
runtime.close()
